In [1]:
import itertools
import json
import sys
import gc
from pathlib import Path
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

2026-02-20 01:20:24.592698: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-20 01:20:24.742242: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-20 01:20:24.807427: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-20 01:20:24.826283: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-20 01:20:24.928865: I tensorflow/core/platform/cpu_feature_guar

In [2]:
EPOCHS = 1000
BATCH_SIZE = 64
ACT_FUNC = tf.nn.relu
MOMENTUM = 0.5

In [3]:
norm_options = ['norm', 'tanh', 'tanh_norm']
hidden_options = [[4096, 2048], [2048, 1024], [4096, 2048, 1024], [2048, 1024, 512], [1024, 1024]]
lr_options = [1e-2, 1e-3, 1e-4, 1e-5]
dropout_options = [(0.0, 0.0), (0.2, 0.5)]
hyperparameter_grid = list(itertools.product(norm_options, hidden_options, lr_options, dropout_options))
print(f"Total no. of hyperparameter combinations: {len(hyperparameter_grid)}")

Total no. of hyperparameter combinations: 120


In [4]:
checkpoint_file = ROOT_DIR / "hyperparam_checkpoint.json"
best_val_loss = np.inf
best_params = None
best_epoch = None
start_idx = 0
if checkpoint_file.exists():
    with open(checkpoint_file, "r") as f:
        checkpoint = json.load(f)
    best_val_loss = checkpoint.get("best_val_loss", np.inf)
    best_params = checkpoint.get("best_params", None)
    best_epoch = checkpoint.get("best_epoch", None)
    start_idx = checkpoint.get("last_completed_idx", -1) + 1
    records = checkpoint.get("records", [])
    print(f"Resuming from index {start_idx}, best_val_loss so far: {best_val_loss}")
else:
    records = []

Resuming from index 94, best_val_loss so far: 280.9870910644531


In [5]:
data_cache = {}
for norm in norm_options:
    train_features, val_features, _, _, train_targets, val_targets, _, _ = load(norm=norm)
    data_cache[norm] = (train_features, val_features, train_targets, val_targets)
    print(f"\n{norm}")
    print("Train features shape:", train_features.shape)
    print("Val features shape:", val_features.shape)
    print("Train targets shape:", train_targets.shape)
    print("Val targets shape:", val_targets.shape)
    print("NaN in train_features:", np.isnan(train_features).any())
    print("Inf in train_features:", np.isinf(train_features).any())
    print("NaN in train_targets:", np.isnan(train_targets).any())
    print("Inf in train_targets:", np.isinf(train_targets).any())
    print("\nFirst 5 rows of train_features:\n", train_features[:5])
    print("First 5 elements of train_targets:\n", train_targets[:5])



norm
Train features shape: (13884, 7060)
Val features shape: (4614, 7060)
Train targets shape: (13884, 1)
Val targets shape: (4614, 1)
NaN in train_features: False
Inf in train_features: False
NaN in train_targets: False
Inf in train_targets: False

First 5 rows of train_features:
 [[-0.38594985 -0.54272044 -0.2307692  ...  0.          0.
   0.        ]
 [-0.38594985 -0.54272044 -0.2307692  ... -0.561097   -0.68233347
   0.07810206]
 [-0.38594985 -0.54272044 -0.2307692  ...  0.4991565   1.9423046
  -0.38811162]
 [-0.38594985 -0.54272044 -0.2307692  ...  0.          0.
   0.        ]
 [-0.38594985 -0.54272044 -0.2307692  ...  0.          0.
   0.        ]]
First 5 elements of train_targets:
 [[ 7.69353  ]
 [ 7.7780533]
 [-1.1985054]
 [ 2.5956845]
 [-5.1399713]]

tanh
Train features shape: (13884, 7060)
Val features shape: (4614, 7060)
Train targets shape: (13884, 1)
Val targets shape: (4614, 1)
NaN in train_features: False
Inf in train_features: False
NaN in train_targets: False
Inf in

In [6]:
def moving_average(x, n):
    return np.convolve(x, np.ones(n) / n, mode='valid')

In [7]:
for idx, (norm_type, hidden_layers, lr, (input_dropout, hidden_dropout)) in enumerate(hyperparameter_grid):
    if idx < start_idx:
        continuessa

    train_features, val_features, train_targets, val_targets = data_cache[norm_type]

    K.clear_session()
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(
                units, input_shape=(train_features.shape[1],), activation=ACT_FUNC, kernel_initializer='he_normal'))
            if input_dropout > 0:
                model.add(Dropout(float(input_dropout)))
        else:
            model.add(Dense(
                units, activation=ACT_FUNC, kernel_initializer='he_normal'))
            if hidden_dropout > 0:
                model.add(Dropout(float(hidden_dropout)))

    model.add(Dense(1, activation='linear', kernel_initializer='he_normal'))
    model.compile(
        loss='mean_squared_error',
        optimizer=SGD(learning_rate=float(lr), momentum=MOMENTUM)
    )
    model.summary()

    history = model.fit(
        train_features, train_targets,
        validation_data=(val_features, val_targets),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=1,
        callbacks=[tf.keras.callbacks.TerminateOnNaN()]
    )

    val_losses = np.array(history.history['val_loss'])
    local_best_epoch = int(np.argmin(val_losses))
    local_best_loss = float(val_losses[local_best_epoch])

    if local_best_loss < best_val_loss:
        best_val_loss = local_best_loss
        best_epoch = local_best_epoch + 1
        best_params = {
            "norm": norm_type,
            "hidden_layers": hidden_layers,
            "learning_rate": lr,
            "input_dropout": input_dropout,
            "hidden_dropout": hidden_dropout,
            "epochs": best_epoch
        }

    records.append({
        "local_params": {
            "norm": norm_type,
            "hidden_layers": hidden_layers,
            "learning_rate": lr,
            "input_dropout": input_dropout,
            "hidden_dropout": hidden_dropout,
        },
        "local_best_epoch": local_best_epoch,
        "local_besNone t_loss": None if np.isnan(local_best_loss) else local_best_loss
    })

    checkpoint_data = {
        "last_completed_idx": idx,
        "best_val_loss": float(best_val_loss),
        "best_params": best_params,
        "best_epoch": best_epoch,
        "records": records
    }
    with open(checkpoint_file, "w") as f:
        json.dump(checkpoint_data, f, indent=2)

    del model
    del history
    K.clear_session()
    gc.collect()
    tf.compat.v1.reset_default_graph()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 2048)              14460928  
                                                                 
 dense_1 (Dense)             (None, 1024)              2098176   
                                                                 
 dense_2 (Dense)             (None, 1)                 1025      
                                                                 
Total params: 16560129 (63.17 MB)
Trainable params: 16560129 (63.17 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


I0000 00:00:1771550453.192010 1564760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771550453.192046 1564760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771550453.192054 1564760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771550453.298958 1564760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771550453.298995 1564760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-02-20

Epoch 1/1000


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

  1/217 [..............................] - ETA: 3:11 - loss: 407.9861

...

217/217 [==============================] - 1s 3ms/step - loss: 459.5140 - val_loss: 378.0108
Epoch 2/1000
217/217 [==============================] - 1s 3ms/step - loss: 406.7953 - val_loss: 373.6342
Epoch 3/1000
217/217 [==============================] - 1s 3ms/step - loss: 384.0146 - val_loss: 346.7403
Epoch 4/1000
217/217 [==============================] - 1s 3ms/step - loss: 362.2934 - val_loss: 345.9773
Epoch 5/1000
217/217 [==============================] - 1s 3ms/step - loss: 348.1510 - val_loss: 345.4494
Epoch 6/1000
217/217 [==============================] - 1s 3ms/step - loss: 332.0540 - val_loss: 340.8802
Epoch 7/1000
217/217 [==============================] - 1s 3ms/step - loss: 323.3944 - val_loss: 347.6735
Epoch 8/1000
217/217 [==============================] - 1s 3ms/step - loss: 314.5523 - val_loss: 341.0601
Epoch 9/1000
217/217 [==============================] - 1s 3ms/step - loss: 305.3081 - val_loss: 339.2077
Epoch 10/1000
217/217 [==============================] - 1s

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

217/217 [==============================] - 1s 3ms/step - loss: 525.5601 - val_loss: 410.0048
Epoch 2/1000
217/217 [==============================] - 1s 3ms/step - loss: 452.7301 - val_loss: 385.8905
Epoch 3/1000
217/217 [==============================] - 1s 3ms/step - loss: 423.3515 - val_loss: 373.3273
Epoch 4/1000
217/217 [==============================] - 1s 3ms/step - loss: 405.7744 - val_loss: 365.1624
Epoch 5/1000
217/217 [==============================] - 1s 3ms/step - loss: 391.5767 - val_loss: 359.3723
Epoch 6/1000
217/217 [==============================] - 1s 3ms/step - loss: 378.7206 - val_loss: 353.1318
Epoch 7/1000
217/217 [==============================] - 1s 3ms/step - loss: 366.7857 - val_loss: 347.4051
Epoch 8/1000
217/217 [==============================] - 1s 3ms/step - loss: 354.3566 - val_loss: 344.5220
Epoch 9/1000
217/217 [==============================] - 1s 3ms/step - loss: 342.9748 - val_loss: 342.2437
Epoch 10/1000
217/217 [==============================] - 1s

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

217/217 [==============================] - 1s 4ms/step - loss: 534.1623 - val_loss: 409.9302
Epoch 2/1000
217/217 [==============================] - 1s 3ms/step - loss: 465.3210 - val_loss: 387.9452
Epoch 3/1000
217/217 [==============================] - 1s 3ms/step - loss: 436.2918 - val_loss: 374.8446
Epoch 4/1000
217/217 [==============================] - 1s 3ms/step - loss: 421.9383 - val_loss: 368.1851
Epoch 5/1000
217/217 [==============================] - 1s 3ms/step - loss: 411.3524 - val_loss: 361.8196
Epoch 6/1000
217/217 [==============================] - 1s 3ms/step - loss: 400.0406 - val_loss: 358.8110
Epoch 7/1000
217/217 [==============================] - 1s 3ms/step - loss: 392.5157 - val_loss: 355.0651
Epoch 8/1000
217/217 [==============================] - 1s 3ms/step - loss: 383.4482 - val_loss: 350.3445
Epoch 9/1000
217/217 [==============================] - 1s 3ms/step - loss: 376.8776 - val_loss: 348.1934
Epoch 10/1000
217/217 [==============================] - 1s

In [8]:
out_file = ROOT_DIR / "best_hyperparams.txt"
with open(out_file, "w") as f:
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"best_val_loss: {best_val_loss}\n")